# 2. U-Net burst detection: training and validation

This independently runnable alternative to `02_tcn_training_and_evaluation.ipynb`
uses a **1D U-Net** to predict one burst probability per time point. It reuses the
same seeded data, train-fitted preprocessing, weighted BCE, validation AP objective,
and event-threshold selection so the development protocol stays comparable.

The workflow includes a tiny-batch training diagnostic, persistent Optuna tuning,
final training, validation plots, and a frozen checkpoint export. The hold-out fold
is never evaluated here. U-Net artifacts have separate filenames from the TCN/CNN/BiGRU run.

Run from the repository environment after `uv sync --extra torch --extra tuning`.
The default budget includes 12 Optuna trials; the configuration below exposes the
training budgets for a shorter exploratory run. Use a new study name/database when
changing the data, search space, or training budget of a persistent study.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from deeplearning_examples.io import load_data
from burst_detection.artifacts import save_holdout_manifest, preprocessor_from_checkpoint
from burst_detection.data import (
    fit_archetype_bank, fit_preprocessor, generate_dataset_splits, targets_to_tensor,
)
from burst_detection.metrics import select_threshold
from burst_detection.model import BurstUNet, count_parameters, create_burst_model
from burst_detection.training import (
    TrainingConfig, artifact_root, load_checkpoint, predict_probabilities,
    set_reproducible_seed, train_model,
)
from burst_detection.tuning import (
    BurstOptunaConfig, best_burst_model_configs, burst_trial_frame,
    optuna_parameter_importances, run_burst_model_search,
)
from burst_detection.visualization import plot_burst_examples, plot_training_history

In [ ]:
SPLIT_CONFIG = {"train_size": 800, "validation_size": 200, "test_size": 200, "random_seed": 42}
ARCHETYPE_SEED = 42
SANITY_SAMPLES = 8
SANITY_EPOCHS = 120
OPTUNA_TOTAL_TRIALS = 12
SEARCH_EPOCHS = 40
FINAL_EPOCHS = 60
STUDY_NAME = "burst-unet-joint-v1"
STUDY_DB_PATH = (artifact_root() / "burst_unet_optuna.db").resolve()
CHECKPOINT_PATH = artifact_root() / "best_unet.pt"
MANIFEST_PATH = artifact_root() / "unet_holdout_models.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {device}")

## Recreate the train and validation folds

Fit background archetypes only on the experimental training trajectories at 100 pM.
Normalization statistics and the positive-class weight are fitted on synthetic
training data only. Inputs contain standardized signal and first difference:
`(batch, 2, 289)`. Dense labels retain their original `(batch, 289)` shape.
The split helper also creates the reproducible test fold, but this notebook never
accesses its signals or labels.

In [ ]:
experimental_train, doses, _ = load_data("train")
archetypes = fit_archetype_bank(
    experimental_train[doses == "100pM"], random_seed=ARCHETYPE_SEED,
)
splits = generate_dataset_splits(archetypes=archetypes, **SPLIT_CONFIG)

preprocessor = fit_preprocessor(splits.train.signals, include_derivative=True)
train_x = preprocessor.transform(splits.train.signals)
validation_x = preprocessor.transform(splits.validation.signals)
train_y = targets_to_tensor(splits.train.labels)
validation_y = targets_to_tensor(splits.validation.labels)
train_x.shape, train_y.shape, preprocessor

## Architecture: temporal encoder, bottleneck, and decoder

Each encoder stage applies two same-padded `Conv1d` layers with `GroupNorm`, `GELU`,
and dropout, then max-pools by two. Channel widths double at each scale. The
bottleneck processes the coarsest features. Each decoder stage upsamples to the
**exact length of its corresponding encoder skip**, concatenates both feature maps,
and applies two more convolutions. A pointwise convolution returns burst logits;
`predict_probabilities` applies sigmoid for evaluation.

For the default depth of three, temporal lengths follow
`289 → 144 → 72 → 36 → 72 → 144 → 289`, while encoder/bottleneck widths are
`16 → 32 → 64 → 128`. Skip connections preserve detailed boundary information
alongside the coarser temporal features. Pooling is internal: neither targets nor
final predictions are downsampled. Inputs must have at least `2**depth` time points.

The decoder uses [PyTorch linear interpolation](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.interpolate.html)
with an explicit target size and `align_corners=False`, which handles the odd
289-sample length without cropping labels. Symmetric convolutions and normalization
across time make this an offline detector that can use future observations.

In [ ]:
set_reproducible_seed(42)
example_model = BurstUNet(input_channels=train_x.shape[1])
with torch.no_grad():
    example_logits = example_model(train_x[:2])
assert example_logits.shape == train_y[:2].shape
pd.Series({
    "input_shape": tuple(train_x[:2].shape),
    "logit_shape": tuple(example_logits.shape),
    "parameters": count_parameters(example_model),
    **example_model.model_config,
}, name="1D U-Net")

## Tiny-batch memorization check

Train a smaller U-Net on eight trajectories and plot the untouched validation
fold as a comparison. The training loss should decrease strongly; separation from
validation loss is expected in a memorization diagnostic. Inspect the actual curve
before concluding that the model can fit the dense targets. This experiment does
not select the final model or its stopping epoch.

In [ ]:
set_reproducible_seed(7)
sanity_model = BurstUNet(
    input_channels=train_x.shape[1], base_channels=8,
    depth=2, kernel_size=5, dropout=0.0,
)
sanity_history = train_model(
    sanity_model, train_x[:SANITY_SAMPLES], train_y[:SANITY_SAMPLES],
    validation_x, validation_y, device=device,
    config=TrainingConfig(
        epochs=SANITY_EPOCHS, batch_size=SANITY_SAMPLES, learning_rate=3e-3,
        weight_decay=0.0, patience=SANITY_EPOCHS, random_seed=7, report_every=30,
    ),
)
plot_training_history(sanity_history, title="U-Net tiny-batch memorization")
display(pd.Series({
    "initial_train_loss": sanity_history.training_loss[0],
    "final_train_loss": sanity_history.training_loss[-1],
    "minimum_validation_loss": min(sanity_history.validation_loss),
    "best_validation_epoch": sanity_history.best_epoch,
}, name="memorization diagnostic"))

## Persistent U-Net hyperparameter search

Optuna tunes base channels (8, 16, 32), pooling depth (2–4), kernel size (3, 5, 7),
and dropout together with learning rate, batch size, optional weight decay,
positive-class-weight cap, and gradient clipping. Weighted binary cross-entropy
uses a train-derived negative/positive ratio. Each trial restores the epoch with
minimum validation loss; its validation average precision is the search objective.

The SQLite study resumes completed work up to the configured **total** trial budget.
All trials share the same data and initialization seed. Changing the depth changes
both capacity and temporal resolution at the bottleneck; extra pooling may remove
short-burst detail, so a deeper U-Net is not assumed to perform better.

In [ ]:
STUDY_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
study = run_burst_model_search(
    "unet", train_x, train_y, validation_x, validation_y, device=device,
    base_training_config=TrainingConfig(
        epochs=SEARCH_EPOCHS, patience=8, random_seed=42, report_every=20,
    ),
    config=BurstOptunaConfig(
        n_trials=OPTUNA_TOTAL_TRIALS, n_trials_mode="total",
        study_name=STUDY_NAME, storage=f"sqlite:///{STUDY_DB_PATH}",
        sampler_seed=42, trial_seed=42,
        n_startup_trials=5, pruning_warmup_epochs=8,
    ),
)

## Inspect the search

Validation AP measures probability ranking without a threshold. Compare it with
validation loss, best epoch, and parameter count to assess convergence and capacity.
Parameter importance describes the sampled configurations and is not causal evidence.

In [ ]:
trials = burst_trial_frame(study, architecture="unet")
display(trials.head(12).round(4))
completed = trials.query("state == 'COMPLETE'").sort_values("trial")
figure, axis = plt.subplots(figsize=(8, 4))
axis.scatter(completed["trial"], completed["validation_ap"], alpha=0.7)
axis.plot(completed["trial"], completed["validation_ap"].cummax(), color="black", label="best so far")
axis.set(title="U-Net search", xlabel="trial", ylabel="validation AP")
axis.grid(alpha=0.2)
axis.legend()
figure.tight_layout()

try:
    importance = optuna_parameter_importances(study)
except (RuntimeError, ValueError):
    importance = pd.Series(dtype=float, name="importance")
if not importance.empty:
    display(importance.round(3))
else:
    print("More completed trial variation is needed to estimate parameter importance.")

## Final training and checkpoint export

Train the selected configuration with the final budget and restore its minimum
validation-loss weights. The checkpoint stores the U-Net model configuration,
training settings, positive weight, and train-fitted preprocessing. The final
validation loss curve determines early stopping; the test fold is not involved.

In [ ]:
model_name, model_config, training_config = best_burst_model_configs(
    study, architecture="unet", final_epochs=FINAL_EPOCHS,
    final_patience=12, report_every=5,
)
set_reproducible_seed(training_config.random_seed)
model = create_burst_model(model_name, model_config)
history = train_model(
    model, train_x, train_y, validation_x, validation_y,
    config=training_config, device=device, checkpoint_path=CHECKPOINT_PATH,
    preprocessor=preprocessor,
)
plot_training_history(history, title="Final U-Net training")
display(pd.Series({
    "model_name": model_name,
    "parameters": count_parameters(model),
    "best_epoch": history.best_epoch,
    **model_config,
}, name="selected U-Net"))

## Freeze the validation operating point

Reload the saved checkpoint to verify that inference can reconstruct the U-Net and
its preprocessing independently. Select the threshold using **validation event F1**.
As in the original notebook, post-processing bridges one-sample gaps and removes
predicted events shorter than three samples. Point and event metrics below are
model-development results, not an unbiased estimate of generalization.

In [ ]:
restored_model, checkpoint = load_checkpoint(CHECKPOINT_PATH, device=device)
restored_preprocessor = preprocessor_from_checkpoint(checkpoint)
validation_probabilities = predict_probabilities(
    restored_model, restored_preprocessor.transform(splits.validation.signals), device=device,
)
selection = select_threshold(splits.validation.labels, validation_probabilities)
validation_results = pd.DataFrame({"unet": {
    "threshold": selection.threshold, **selection.metrics.to_dict(),
}}).T
display(validation_results.round(4))

## Validation signal, probabilities, and events

Green regions mark true bursts, red regions mark predicted events, and blue curves
show burst probabilities with the selected threshold. Inspect missed short events,
boundary shifts, and fragmented predictions alongside the aggregate metrics.

In [ ]:
plot_burst_examples(
    splits.validation.signals, splits.validation.labels,
    probabilities=validation_probabilities, threshold=selection.threshold,
    time_minutes=splits.validation.time_minutes, maximum_examples=4,
);

In [ ]:
manifest_path = save_holdout_manifest(
    MANIFEST_PATH,
    candidates=[{
        "architecture": "unet",
        "model_name": model_name,
        "checkpoint": CHECKPOINT_PATH.name,
        "threshold": selection.threshold,
        "validation_metrics": selection.metrics.to_dict(),
        "parameter_count": count_parameters(restored_model),
        "best_epoch": history.best_epoch,
    }],
    split_config={**SPLIT_CONFIG, "archetype_seed": ARCHETYPE_SEED},
)
manifest_path

## Continue with the frozen hold-out evaluation

The run writes `best_unet.pt`, `burst_unet_optuna.db`, and
`unet_holdout_models.json` under `notebooks/artifacts/`.

Run `02_tcn_training_and_evaluation.ipynb` as well, then open
`03_holdout_test_and_model_comparison.ipynb`. Notebook 03 loads both frozen manifests,
checks that their split and selection protocols agree, and compares TCN, CNN, BiGRU,
U-Net, and the classical detector together. No manifest-path change is needed.

Freeze all candidate choices before inspecting any hold-out results. Validation
scores from this notebook and the original `02_` notebook share the same default
split, but they do not establish which architecture generalizes best. These are
synthetic bursts on experimental backgrounds; real annotated trajectories require
separate validation.